In [2]:
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from datasets import load_dataset
from chronos import ChronosPipeline
sys.path.append('.')
from ordinal_head import OrdinalCrossEntropyLoss

In [3]:
def smape(actual, forecast):
    return 100 * np.mean(2 * np.abs(actual - forecast) / (np.abs(actual) + np.abs(forecast) + 1e-8))

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

pipeline_original = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.float32,
)

Device: cpu


`torch_dtype` is deprecated! Use `dtype` instead!


In [5]:
dataset = load_dataset("autogluon/chronos_datasets", "m4_hourly", split="train")
df = dataset.to_pandas()

results_original = []
for series_id in df['id'].unique()[:20]:
    series_data = df[df['id'] == series_id]['target'].values[0]
    if len(series_data) < 100:
        continue

    train = series_data[:-48]
    test = series_data[-48:]

    forecast = pipeline_original.predict(
        torch.tensor(train, dtype=torch.float32),
        prediction_length=48,
        num_samples=20
    )
    forecast_np = forecast.numpy()
    if forecast_np.ndim == 3:
        forecast_np = forecast_np[0]
    forecast_med = np.median(forecast_np, axis=0)

    results_original.append({
        'series_id': series_id,
        'mae': mean_absolute_error(test, forecast_med),
        'smape': smape(test, forecast_med)
    })

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating train split: 100%|███████████████████████████████████████████████| 414/414 [00:00<00:00, 7717.76 examples/s]


In [6]:
print("Средние результаты оригинального Chronos:")
print(f"MAE: {np.mean([r['mae'] for r in results_original]):.3f}")
print(f"sMAPE: {np.mean([r['smape'] for r in results_original]):.2f}%")

Средние результаты оригинального Chronos:
MAE: 501.197
sMAPE: 4.20%
